In [1]:
# Imports

import helpers.helper_functions as hf
import mne
import os.path as op
from mne.channels import combine_channels
import pandas as pd
from mne.beamformer import make_lcmv, apply_lcmv_epochs
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
from scipy.signal import hilbert


ss = hf.settings_dict()

In [2]:
# Important variables
tmin = 0.7
tmax = 3.7

In [3]:
for subject_index in ss['subject_idx_list']:
    # loop over each event type
    for event_id in ss['event_id_list']:
        event_name = str(event_id)
        subjects_dir = ss['fs_subjects_dir']
        subject = ss['subject_id_list'][subject_index]
        print("loading dataset for subject: ", subject)

        save_dir = Path(ss['hilbert_dir']) / subject / event_name
        save_dir.mkdir(parents=True, exist_ok=True)

        stc_file = Path(ss['stc_dir']) / subject / event_name / f"{subject}-event-{event_name}-vol.stc"

        stc = mne.read_source_estimate(stc_file)
        cropped_stc = stc.copy().crop(tmin=tmin, tmax=tmax)

        # hilbert transform the stc

        analytic_signal = hilbert(cropped_stc.data, axis=1)
        phase = np.angle(analytic_signal).astype(float)
        amplitude = np.abs(analytic_signal)

        mean_amp = np.mean(amplitude, axis=1)

        # pick the voxel with the highest mean amplitude to be used as a reference instead of the retina one
        ref_voxel = np.argmax(mean_amp)

        print(f"Picking voxel {ref_voxel} as a reference with a mean amplitude of {mean_amp[ref_voxel]}.")

        # save the hilbert transformations in a new stc

        hil_stc = cropped_stc.copy()
        hil_stc.data = phase

        hil_stc.save(save_dir / f"{subject}-event-{event_name}-hilbert-vol.stc" , overwrite=True)

        #TODO: add masking based on the amplitude of the signals

        # now save the hilbert transformed references

        times = cropped_stc.times
        voxel_phase = phase[ref_voxel, :]

        # read the retina csv
        retina_file = Path(ss['stc_dir']) / subject / event_name / f"{subject}-event-{event_name}-retina.csv"

        retina_df = pd.read_csv(retina_file)

        retina_df_cropped = retina_df[(retina_df["time_s"] >= tmin) & (retina_df["time_s"] <= tmax)]

        analytic_signal_retina = hilbert(retina_df_cropped["amplitude"])
        retina_phase = np.angle(analytic_signal_retina).astype(float)

        # Create dataframe
        df = pd.DataFrame({
            "time_s": times,
            "retina_phase": retina_phase,
            f"{str(ref_voxel)}_voxel_phase": voxel_phase,
        })

        # Save to CSV
        df.to_csv(save_dir / f"{subject}-event-{event_name}-reference.csv" , index=False)





loading dataset for subject:  0005_3SJ
Picking voxel 477 as a reference with a mean amplitude of 3.1716372966766357.
Writing STC to disk...
[done]
loading dataset for subject:  0002_TCZ
Picking voxel 1317 as a reference with a mean amplitude of 1.7553266286849976.
Writing STC to disk...
[done]
